Edits to add tuples for UK

In [2]:
import re
import unicodedata
import pandas as pd

# -----------------------------
# 1) Reference dictionaries
# -----------------------------
US_STATE_ALIASES = {
    # Full names
    'alabama':'Alabama', 'alaska':'Alaska', 'arizona':'Arizona', 'arkansas':'Arkansas',
    'california':'California', 'colorado':'Colorado', 'connecticut':'Connecticut',
    'delaware':'Delaware', 'florida':'Florida', 'georgia':'Georgia', 'hawaii':'Hawaii',
    'idaho':'Idaho', 'illinois':'Illinois', 'indiana':'Indiana', 'iowa':'Iowa',
    'kansas':'Kansas', 'kentucky':'Kentucky', 'louisiana':'Louisiana', 'maine':'Maine',
    'maryland':'Maryland', 'massachusetts':'Massachusetts', 'michigan':'Michigan',
    'minnesota':'Minnesota', 'mississippi':'Mississippi', 'missouri':'Missouri',
    'montana':'Montana', 'nebraska':'Nebraska', 'nevada':'Nevada', 'new hampshire':'New Hampshire',
    'new jersey':'New Jersey', 'new mexico':'New Mexico', 'new york':'New York',
    'north carolina':'North Carolina', 'north dakota':'North Dakota', 'ohio':'Ohio',
    'oklahoma':'Oklahoma', 'oregon':'Oregon', 'pennsylvania':'Pennsylvania',
    'rhode island':'Rhode Island', 'south carolina':'South Carolina', 'south dakota':'South Dakota',
    'tennessee':'Tennessee', 'texas':'Texas', 'utah':'Utah', 'vermont':'Vermont',
    'virginia':'Virginia', 'washington':'Washington', 'west virginia':'West Virginia',
    'wisconsin':'Wisconsin', 'wyoming':'Wyoming',
    # DC
    'district of columbia':'District of Columbia', 'washington dc':'District of Columbia', 'washington, d c':'District of Columbia',
    # Abbreviations
    'al':'Alabama','ak':'Alaska','az':'Arizona','ar':'Arkansas','ca':'California','co':'Colorado','ct':'Connecticut',
    'de':'Delaware','fl':'Florida','ga':'Georgia','hi':'Hawaii','id':'Idaho','il':'Illinois','in':'Indiana','ia':'Iowa',
    'ks':'Kansas','ky':'Kentucky','la':'Louisiana','me':'Maine','md':'Maryland','ma':'Massachusetts','mi':'Michigan',
    'mn':'Minnesota','ms':'Mississippi','mo':'Missouri','mt':'Montana','ne':'Nebraska','nv':'Nevada','nh':'New Hampshire',
    'nj':'New Jersey','nm':'New Mexico','ny':'New York','nc':'North Carolina','nd':'North Dakota','oh':'Ohio',
    'ok':'Oklahoma','or':'Oregon','pa':'Pennsylvania','ri':'Rhode Island','sc':'South Carolina','sd':'South Dakota',
    'tn':'Tennessee','tx':'Texas','ut':'Utah','vt':'Vermont','va':'Virginia','wa':'Washington','wv':'West Virginia',
    'wi':'Wisconsin','wy':'Wyoming'
}

US_TERRITORY_TO_STATE_HINT = {
    'utah territory':'Utah', 'arizona territory':'Arizona', 'idaho territory':'Idaho',
    'wyoming territory':'Wyoming', 'new mexico territory':'New Mexico', 'nebraska territory':'Nebraska',
    'iowa territory':'Iowa', 'oregon territory':'Oregon', 'colorado territory':'Colorado',
    'nevada territory':'Nevada', 'deseret':'Utah',
}

COUNTRY_ALIASES = {
    # USA
    'united states':'United States', 'united states of america':'United States', 'usa':'United States',
    'u.s.a.':'United States','u.s.':'United States','us':'United States',
    'united states territory':'United States', 'untied states':'United States', 'united sates':'United States',
    'united staes':'United States', 'united sttes':'United States',
    # UK & parts (note: we normalize the *country* to UK; nations handled separately below)
    'united kingdom':'United Kingdom','great britain':'United Kingdom','uk':'United Kingdom','britain':'United Kingdom',
    'england':'United Kingdom','scotland':'United Kingdom','wales':'United Kingdom','northern ireland':'United Kingdom',
    # Ireland (Republic)
    'ireland':'Ireland',
    # Netherlands
    'holland':'Netherlands','the netherlands':'Netherlands','netherland':'Netherlands',
    # Mexico
    'old mexico':'Mexico',
    # Canada historical
    'upper canada':'Canada','canada west':'Canada','canada east':'Canada','british colonial america':'United States',
    # Germany/Prussia historical
    'prussia':'Germany','east prussia':'Germany','german empire':'Germany','hanover':'Germany',
    # Misc
    'sandwich islands':'Hawaii','hawaii':'United States',
    'holland, netherlands':'Netherlands',
}

SUBDIVISION_HINTS_TO_COUNTRY = {
    # Canada
    'ontario':'Canada','quebec':'Canada','alberta':'Canada','british columbia':'Canada',
    'manitoba':'Canada','saskatchewan':'Canada','new brunswick':'Canada','nova scotia':'Canada',
    'prince edward island':'Canada','pei':'Canada','newfoundland':'Canada','labrador':'Canada',
    # Mexico
    'chihuahua':'Mexico','sonora':'Mexico','coahuila':'Mexico','nuevo leon':'Mexico','durango':'Mexico',
    'tamaulipas':'Mexico','zacatecas':'Mexico','jalisco':'Mexico','puebla':'Mexico',
    # Australia
    'new south wales':'Australia','n s w':'Australia','victoria':'Australia',
    'queensland':'Australia','south australia':'Australia','western australia':'Australia','tasmania':'Australia',
    # New Zealand
    'otago':'New Zealand','manawatu':'New Zealand','northland':'New Zealand','auckland':'New Zealand',
}

# NEW: UK nation detection
UK_NATIONS = {'england':'England', 'scotland':'Scotland', 'wales':'Wales', 'northern ireland':'Northern Ireland'}

# Common historic counties/areas to a UK nation (covers many seen in your file)
UK_SUBDIVISION_TO_NATION = {
    # England
    'middlesex':'England','lancashire':'England','yorkshire':'England','derbyshire':'England',
    'gloucestershire':'England','kent':'England','somerset':'England','wiltshire':'England',
    'cornwall':'England','cheshire':'England','nottinghamshire':'England','warwickshire':'England',
    'staffordshire':'England','lincolnshire':'England','rutland':'England','norfolk':'England',
    'suffolk':'England','cambridgeshire':'England','oxfordshire':'England','berkshire':'England',
    'leicestershire':'England','herefordshire':'England','shropshire':'England','worcestershire':'England',
    'hampshire':'England','essex':'England','bedfordshire':'England','hertfordshire':'England',
    'northumberland':'England','durham':'England','cumberland':'England','westmorland':'England',
    'surrey':'England','sussex':'England','devon':'England','dorset':'England','buckinghamshire':'England',
    'isle of man':'England',  # dataset sometimes treats this UK-adjacent; adjust if you prefer separate treatment
    # Wales
    'glamorgan':'Wales','monmouth':'Wales','monmouthshire':'Wales','pembrokeshire':'Wales',
    'cardiganshire':'Wales','carmarthenshire':'Wales','denbighshire':'Wales','flintshire':'Wales',
    'merionethshire':'Wales','montgomeryshire':'Wales','radnorshire':'Wales','anglesey':'Wales',
    # Scotland
    'ayrshire':'Scotland','lanarkshire':'Scotland','dunbartonshire':'Scotland','renfrewshire':'Scotland',
    'fife':'Scotland','banffshire':'Scotland','aberdeenshire':'Scotland','perthshire':'Scotland',
    'argyllshire':'Scotland','moray':'Scotland','ross-shire':'Scotland','sutherland':'Scotland',
    'caithness':'Scotland','wigtownshire':'Scotland','kirkcudbrightshire':'Scotland','stirlingshire':'Scotland',
    'selkirkshire':'Scotland','roxburghshire':'Scotland','peeblesshire':'Scotland','berwickshire':'Scotland',
    # Northern Ireland (counties)
    'antrim':'Northern Ireland','armagh':'Northern Ireland','down':'Northern Ireland',
    'fermanagh':'Northern Ireland','londonderry':'Northern Ireland','tyrone':'Northern Ireland',
}

MOJIBAKE_FIXES = {
    'ÃƒÂ‚Ã‚Â':'', 'ÃƒÂƒÃ‚ÂƒÃƒÂ†Ã‚Â’ÃƒÂƒÃ‚Â‚ÃƒÂ‚Ã‚Â':'', 'ÃƒÂƒÃ‚Âƒ':'', 'ÃƒÂ†Ã‚Â’':'',
    'ÃƒÂ‚Ã‚Â':'', 'ÃƒÂ¢Ã‚Â€Ã‚Â“':'-', 'ÃƒÂ…Ã‚Â¸':'y',
}

# -----------------------------
# 2) Helper functions
# -----------------------------
def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ''
    for bad, good in MOJIBAKE_FIXES.items():
        text = text.replace(bad, good)
    text = unicodedata.normalize('NFKC', text).strip()
    text = re.sub(r'[|;]+', ',', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def to_lower(s: str) -> str:
    return s.lower().strip()

def norm_country_name(tok: str) -> str | None:
    t = to_lower(tok)
    if t in COUNTRY_ALIASES:
        return COUNTRY_ALIASES[t]
    t2 = re.sub(r'^(kingdom|republic|state|states|duchy|grand duchy|empire|commonwealth) of ', '', t)
    if t2 in COUNTRY_ALIASES:
        return COUNTRY_ALIASES[t2]
    known = {
        'united states','united kingdom','canada','mexico','germany','sweden','denmark','norway','finland',
        'iceland','netherlands','belgium','france','italy','spain','portugal','switzerland','austria',
        'russia','poland','czech republic','slovakia','hungary','romania','bulgaria','greece','ireland',
        'ukraine','lithuania','latvia','estonia','croatia','slovenia','serbia','bosnia and herzegovina',
        'montenegro','north macedonia','albania','turkey','china','japan','korea','india','pakistan',
        'bangladesh','sri lanka','nepal','myanmar','thailand','vietnam','laos','cambodia','malaysia',
        'singapore','indonesia','philippines','australia','new zealand','south africa','egypt','israel',
        'lebanon','syria','jordan','iraq','iran','saudi arabia','yemen','uae','oman','qatar','bahrain',
        'kuwait','morocco','algeria','tunisia','ethiopia','kenya','tanzania','uganda','nigeria','ghana',
        'brazil','argentina','chile','peru','colombia','venezuela','uruguay','paraguay','bolivia',
        'ecuador','guyana','suriname','panama','costa rica','nicaragua','honduras','guatemala','el salvador',
        'cuba','dominican republic','haiti'
    }
    if t2 in known:
        return t2.title() if t2 != 'united states' else 'United States'
    return None

def is_us_country_token(tok: str) -> bool:
    return norm_country_name(tok) == 'United States'

def extract_tokens(raw: str) -> list[str]:
    s = basic_clean(raw)
    if ',' in s:
        parts = [p.strip() for p in s.split(',') if p.strip()]
    else:
        parts = re.split(r'\s{1,}', s)
    return parts

def normalize_list_tokens(parts: list[str]) -> list[str]:
    out = []
    for p in parts:
        p_low = to_lower(p)
        p_low = (p_low.replace('untied states', 'united states')
                      .replace('unites states', 'united states')
                      .replace('united sates', 'united states')
                      .replace('united staes', 'united states')
                      .replace('united sttes', 'united states')
                      .replace('u s a', 'usa').replace('u. s. a.', 'usa').replace('u.s.', 'us')
                      .replace('territory of ', '').replace(' ,', ',').strip())
        out.append(p_low)
    return out

def first_true(iterable, pred):
    for x in iterable:
        if pred(x):
            return x
    return None

# -----------------------------
# 3) Core normalization logic (UPDATED FOR UK TUPLES)
# -----------------------------
def clean_birthplace_value(raw: str):
    """
    Input: birthplace string.
    Output:
      - 'Country' (str) if outside United States and not a UK nation
      - (state, 'United States') if inside U.S.
      - (nation, 'United Kingdom') if inside UK (England/Scotland/Wales/Northern Ireland)
      - 'Ireland' (str) for Republic of Ireland
      - None if indeterminate
    """
    if not isinstance(raw, str) or not raw.strip():
        return None

    parts = extract_tokens(raw)
    parts = normalize_list_tokens(parts)

    # 1) Look for explicit country tokens (from the end)
    explicit_country = None
    for p in reversed(parts):
        c = norm_country_name(p)
        if c:
            explicit_country = c
            break

    # 2) Detect explicit UK nation tokens anywhere (england, scotland, etc.)
    explicit_uk_nation = None
    for p in parts:
        if p in UK_NATIONS:
            explicit_uk_nation = UK_NATIONS[p]
            break

    # 3) U.S. territory/state hints
    territory_token = first_true(parts, lambda t: t.endswith(' territory'))
    us_territory_state_hint = None
    if territory_token and territory_token in US_TERRITORY_TO_STATE_HINT:
        us_territory_state_hint = US_TERRITORY_TO_STATE_HINT[territory_token]

    state_token = None
    for p in reversed(parts):
        if p.endswith(' territory'):
            stem = p.replace(' territory', '').strip()
            if stem in US_TERRITORY_TO_STATE_HINT:
                state_token = US_TERRITORY_TO_STATE_HINT[p]
                break
            p_state_candidate = stem
        else:
            p_state_candidate = p

        if p_state_candidate in US_STATE_ALIASES:
            state_token = US_STATE_ALIASES[p_state_candidate]
            break
        if p_state_candidate.title().lower() in US_STATE_ALIASES:
            state_token = US_STATE_ALIASES[p_state_candidate.title().lower()]
            break

    if not state_token and us_territory_state_hint:
        state_token = us_territory_state_hint

    # 4) Non-US inference via subdivisions (Canada, Mexico, ANZ)
    inferred_country = None
    if not explicit_country:
        for p in reversed(parts):
            if p in SUBDIVISION_HINTS_TO_COUNTRY:
                inferred_country = SUBDIVISION_HINTS_TO_COUNTRY[p]
                break

    # 5) UK nation inference from historic counties/areas
    inferred_uk_nation = None
    for p in reversed(parts):
        if p in UK_SUBDIVISION_TO_NATION:
            inferred_uk_nation = UK_SUBDIVISION_TO_NATION[p]
            break

    # ---------------- Resolution ----------------
    # US cases
    if explicit_country == 'United States' or state_token:
        if not state_token:
            return ('Unknown', 'United States')
        return (state_token, 'United States')

    # UK nation explicit (overrides generic UK country)
    if explicit_uk_nation:
        return (explicit_uk_nation, 'United Kingdom')

    # Republic of Ireland explicit
    if explicit_country == 'Ireland':
        return 'Ireland'

    # Explicit "United Kingdom" but we can infer a nation from county
    if explicit_country == 'United Kingdom' and inferred_uk_nation:
        return (inferred_uk_nation, 'United Kingdom')

    # No explicit country, but we inferred a UK nation from county
    if not explicit_country and inferred_uk_nation:
        return (inferred_uk_nation, 'United Kingdom')

    # Other explicit non-US countries
    if explicit_country and explicit_country != 'United Kingdom':
        return explicit_country

    # Other inferred countries (e.g., Canada/Mexico/Australia/NZ)
    if inferred_country:
        return inferred_country

    # Heuristic: last token might be the country or UK nation word
    if parts:
        last_guess_country = norm_country_name(parts[-1])
        if last_guess_country == 'United Kingdom':
            # still try to infer nation from earlier tokens
            if inferred_uk_nation:
                return (inferred_uk_nation, 'United Kingdom')
            # fall back: unknown nation within UK
            return ('Unknown UK nation', 'United Kingdom')
        if last_guess_country and last_guess_country != 'United States':
            if last_guess_country == 'Ireland':
                return 'Ireland'
            return last_guess_country

    return None

# -----------------------------
# 4) Batch application helpers
# -----------------------------
def standardize_birthplaces(df: pd.DataFrame, col: str = 'birth_place') -> pd.DataFrame:
    df = df.copy()
    df['standardized_birthplace'] = df[col].apply(clean_birthplace_value)
    
    # break into 2 columns
    data_dict = {"state": [], "country": []}
    
    for i in range(len(df)):
        cur_birthplace = df['standardized_birthplace'].iloc[i]
        cur_birthplace = str(cur_birthplace)
        
        if "'" in cur_birthplace and "(" in cur_birthplace:
            vals = cur_birthplace.split("'")
            state = vals[1]
            country = vals[-2]
        else:
            state = ""
            country = cur_birthplace
        data_dict["state"].append(state)
        data_dict["country"].append(country)
    df["state"] = data_dict["state"]  
    df["country"] = data_dict["country"]
    df = df.drop(columns=['standardized_birthplace'])
    
    return df

# -----------------------------
# 5) Example usage
# -----------------------------
if __name__ == '__main__':
    CSV_PATH = 'parent2_cleaning.csv'
    try:
        df_raw = pd.read_csv(CSV_PATH)
        if df_raw.shape[1] == 1 and df_raw.columns[0] != 'birth_place':
            df_raw.columns = ['birth_place']
        df_clean = standardize_birthplaces(df_raw, 'birth_place')
        print(df_clean.head(20).to_string(index=False))
        df_clean.to_csv('parent2_cleaned.csv', index=False)
    except Exception as e:
        print("Failed to process CSV:", e)

                                         birth_place        state        country
             Old Beaver, Clark, Idaho, United States        Idaho  United States
Charlestown, Washington, Rhode Island, United States Rhode Island  United States
              Wellsville, Cache, Utah, United States         Utah  United States
              Fillmore, Millard, Utah, United States         Utah  United States
                         Asnæs, Ods, Holbæk, Denmark                     Denmark
      Salt Lake City, Salt Lake, Utah, United States         Utah  United States
           Millcreek, Salt Lake, Utah, United States         Utah  United States
     Breightmet, Lancashire, England, United Kingdom      England United Kingdom
      Bingham Canyon, Salt Lake, Utah, United States         Utah  United States
         Washington, Washington, Utah, United States         Utah  United States
              Coalville, Summit, Utah, United States         Utah  United States
                   Hyrum, Ca

2019 ('Utah', 'United States')
2020 ('Idaho', 'United States')
2021 ('Utah', 'United States')
2022 ('Idaho', 'United States')
2023 ('Utah', 'United States')
2024 ('Arizona', 'United States')
2025 ('Utah', 'United States')
2026 ('Utah', 'United States')
2027 ('Utah', 'United States')
2028 ('Utah', 'United States')
2029 ('Utah', 'United States')
2030 ('Utah', 'United States')
2031 Canada
2032 ('Wales', 'United Kingdom')
2033 ('Utah', 'United States')
2034 ('Utah', 'United States')
2035 ('Idaho', 'United States')
2036 ('Utah', 'United States')
2037 ('Utah', 'United States')
2038 ('England', 'United Kingdom')
2039 ('Utah', 'United States')
2040 ('Utah', 'United States')
2041 ('Utah', 'United States')
2042 ('Utah', 'United States')
2043 ('England', 'United Kingdom')
2044 ('Utah', 'United States')
2045 ('England', 'United Kingdom')
2046 ('Utah', 'United States')
2047 ('Utah', 'United States')
2048 ('Utah', 'United States')
2049 Netherlands
2050 ('Utah', 'United States')
2051 ('Utah', 'United

3416 ('Utah', 'United States')
3417 ('Utah', 'United States')
3418 Germany
3419 ('Arizona', 'United States')
3420 ('Arizona', 'United States')
3421 ('Arizona', 'United States')
3422 ('Arizona', 'United States')
3423 ('Idaho', 'United States')
3424 ('Utah', 'United States')
3425 ('Utah', 'United States')
3426 ('Utah', 'United States')
3427 ('England', 'United Kingdom')
3428 ('Utah', 'United States')
3429 ('Idaho', 'United States')
3430 ('Utah', 'United States')
3431 ('Utah', 'United States')
3432 ('Iowa', 'United States')
3433 ('Utah', 'United States')
3434 ('Utah', 'United States')
3435 ('Utah', 'United States')
3436 ('Utah', 'United States')
3437 ('Utah', 'United States')
3438 Sweden
3439 ('Idaho', 'United States')
3440 ('Idaho', 'United States')
3441 Denmark
3442 Denmark
3443 Denmark
3444 ('Utah', 'United States')
3445 ('Utah', 'United States')
3446 ('Utah', 'United States')
3447 ('Idaho', 'United States')
3448 ('Utah', 'United States')
3449 ('Utah', 'United States')
3450 ('Nevada', 

4710 ('Idaho', 'United States')
4711 Sweden
4712 ('Arizona', 'United States')
4713 ('Michigan', 'United States')
4714 ('Utah', 'United States')
4715 Sweden
4716 ('Idaho', 'United States')
4717 ('Utah', 'United States')
4718 ('Iowa', 'United States')
4719 ('Utah', 'United States')
4720 ('Utah', 'United States')
4721 ('Utah', 'United States')
4722 ('Utah', 'United States')
4723 ('Utah', 'United States')
4724 ('Utah', 'United States')
4725 ('Idaho', 'United States')
4726 ('Utah', 'United States')
4727 ('Utah', 'United States')
4728 ('Utah', 'United States')
4729 ('Utah', 'United States')
4730 ('Utah', 'United States')
4731 ('Utah', 'United States')
4732 ('Utah', 'United States')
4733 ('Utah', 'United States')
4734 ('Utah', 'United States')
4735 ('Utah', 'United States')
4736 ('Idaho', 'United States')
4737 ('Utah', 'United States')
4738 ('Missouri', 'United States')
4739 ('Missouri', 'United States')
4740 ('Missouri', 'United States')
4741 ('Missouri', 'United States')
4742 ('Utah', 'Unit

6018 ('Virginia', 'United States')
6019 ('Idaho', 'United States')
6020 ('Utah', 'United States')
6021 ('Utah', 'United States')
6022 ('Utah', 'United States')
6023 ('Arizona', 'United States')
6024 ('Idaho', 'United States')
6025 ('England', 'United Kingdom')
6026 ('Utah', 'United States')
6027 ('Utah', 'United States')
6028 ('Idaho', 'United States')
6029 ('Utah', 'United States')
6030 ('England', 'United Kingdom')
6031 ('Utah', 'United States')
6032 ('Utah', 'United States')
6033 ('Utah', 'United States')
6034 ('Idaho', 'United States')
6035 ('Utah', 'United States')
6036 ('Utah', 'United States')
6037 ('Utah', 'United States')
6038 ('Ohio', 'United States')
6039 Denmark
6040 Denmark
6041 ('Idaho', 'United States')
6042 ('Utah', 'United States')
6043 ('Utah', 'United States')
6044 ('Utah', 'United States')
6045 ('Utah', 'United States')
6046 ('Utah', 'United States')
6047 ('Utah', 'United States')
6048 ('Utah', 'United States')
6049 ('Wyoming', 'United States')
6050 ('Utah', 'United

...

43152 ('Utah', 'United States')
43153 ('England', 'United Kingdom')
43154 ('Utah', 'United States')
43155 ('Utah', 'United States')
43156 ('Utah', 'United States')
43157 ('Utah', 'United States')
43158 ('Utah', 'United States')
43159 ('Utah', 'United States')
43160 Norway
43161 ('Arizona', 'United States')
43162 ('Utah', 'United States')
43163 ('Utah', 'United States')
43164 Sweden
43165 ('Utah', 'United States')
43166 ('Utah', 'United States')
43167 ('Utah', 'United States')
43168 ('Utah', 'United States')
43169 Canada
43170 Sweden
43171 Sweden
43172 ('Utah', 'United States')
43173 ('Utah', 'United States')
43174 Germany
43175 ('Utah', 'United States')
43176 ('Utah', 'United States')
43177 ('Utah', 'United States')
43178 ('Utah', 'United States')
43179 ('Utah', 'United States')
43180 Mexico
43181 ('Utah', 'United States')
43182 ('Ohio', 'United States')
43183 ('Ohio', 'United States')
43184 ('Utah', 'United States')
43185 ('England', 'United Kingdom')
43186 ('Utah', 'United States')
4

44260 ('Idaho', 'United States')
44261 ('Scotland', 'United Kingdom')
44262 ('Idaho', 'United States')
44263 ('Utah', 'United States')
44264 ('Utah', 'United States')
44265 Switzerland
44266 Norway
44267 Norway
44268 ('Utah', 'United States')
44269 ('Idaho', 'United States')
44270 ('Utah', 'United States')
44271 ('Utah', 'United States')
44272 ('Utah', 'United States')
44273 ('Idaho', 'United States')
44274 ('Idaho', 'United States')
44275 ('Idaho', 'United States')
44276 ('Arizona', 'United States')
44277 ('Utah', 'United States')
44278 ('New York', 'United States')
44279 ('New York', 'United States')
44280 ('New York', 'United States')
44281 ('New York', 'United States')
44282 ('Utah', 'United States')
44283 ('Utah', 'United States')
44284 ('Utah', 'United States')
44285 Australia
44286 ('England', 'United Kingdom')
44287 Netherlands
44288 Netherlands
44289 Netherlands
44290 Netherlands
44291 Sweden
44292 ('Idaho', 'United States')
44293 Sweden
44294 ('Idaho', 'United States')
44295 

45238 ('Utah', 'United States')
45239 ('Utah', 'United States')
45240 ('Utah', 'United States')
45241 ('Utah', 'United States')
45242 ('Utah', 'United States')
45243 ('Utah', 'United States')
45244 ('Utah', 'United States')
45245 ('Utah', 'United States')
45246 ('Idaho', 'United States')
45247 ('Utah', 'United States')
45248 ('Utah', 'United States')
45249 ('Arizona', 'United States')
45250 ('Utah', 'United States')
45251 ('Utah', 'United States')
45252 Sweden
45253 ('England', 'United Kingdom')
45254 ('England', 'United Kingdom')
45255 ('England', 'United Kingdom')
45256 ('Utah', 'United States')
45257 ('Idaho', 'United States')
45258 ('Idaho', 'United States')
45259 ('Utah', 'United States')
45260 ('Missouri', 'United States')
45261 ('Utah', 'United States')
45262 ('Utah', 'United States')
45263 ('Utah', 'United States')
45264 ('Utah', 'United States')
45265 ('Utah', 'United States')
45266 ('Utah', 'United States')
45267 ('Utah', 'United States')
45268 ('Utah', 'United States')
45269

46376 ('Idaho', 'United States')
46377 ('Utah', 'United States')
46378 ('Utah', 'United States')
46379 ('Utah', 'United States')
46380 ('Idaho', 'United States')
46381 ('Arizona', 'United States')
46382 ('Utah', 'United States')
46383 ('Utah', 'United States')
46384 ('Utah', 'United States')
46385 ('Utah', 'United States')
46386 ('Utah', 'United States')
46387 ('Idaho', 'United States')
46388 ('Utah', 'United States')
46389 ('Arizona', 'United States')
46390 Germany
46391 ('Utah', 'United States')
46392 ('Utah', 'United States')
46393 ('Utah', 'United States')
46394 ('Oregon', 'United States')
46395 ('Wyoming', 'United States')
46396 ('New York', 'United States')
46397 ('Utah', 'United States')
46398 ('Utah', 'United States')
46399 ('Utah', 'United States')
46400 ('Utah', 'United States')
46401 ('Utah', 'United States')
46402 ('Utah', 'United States')
46403 ('Utah', 'United States')
46404 ('Utah', 'United States')
46405 ('Idaho', 'United States')
46406 Netherlands
46407 ('Utah', 'Unite

47565 ('Utah', 'United States')
47566 ('Illinois', 'United States')
47567 ('Utah', 'United States')
47568 ('Arizona', 'United States')
47569 ('Utah', 'United States')
47570 ('Arizona', 'United States')
47571 ('Utah', 'United States')
47572 ('Idaho', 'United States')
47573 ('Utah', 'United States')
47574 ('Utah', 'United States')
47575 ('Utah', 'United States')
47576 ('Utah', 'United States')
47577 Denmark
47578 ('Scotland', 'United Kingdom')
47579 ('Wales', 'United Kingdom')
47580 ('Utah', 'United States')
47581 ('Utah', 'United States')
47582 ('Idaho', 'United States')
47583 ('Utah', 'United States')
47584 ('Utah', 'United States')
47585 ('Utah', 'United States')
47586 ('Utah', 'United States')
47587 Denmark
47588 ('Utah', 'United States')
47589 ('Utah', 'United States')
47590 ('Utah', 'United States')
47591 ('Utah', 'United States')
47592 ('Utah', 'United States')
47593 ('Utah', 'United States')
47594 ('Utah', 'United States')
47595 ('Utah', 'United States')
47596 ('Kansas', 'United 

# Data Validation

Let's check on the AI output

In [30]:
df = pd.read_csv("parent2_no_duplicates.csv")

df.head()

,birth_place,state,country
0,"Old Beaver, Clark, Idaho, United States",Idaho,United States
1,"Charlestown, Washington, Rhode Island, United ...",Rhode Island,United States
2,"Wellsville, Cache, Utah, United States",Utah,United States
3,"Fillmore, Millard, Utah, United States",Utah,United States
4,"Asnæs, Ods, Holbæk, Denmark",NaN,Denmark


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10049 entries, 0 to 10048
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   birth_place  10049 non-null  object
 1   state        6132 non-null   object
 2   country      10047 non-null  object
dtypes: object(3)
memory usage: 235.7+ KB


In [23]:
df.drop_duplicates(inplace = True)

In [24]:
print(len(df))

10049


In [28]:
df['country'] = df['country'].apply(lambda x:  str(x).strip())

In [35]:
df.to_csv("parent2_no_duplicates.csv", index = False)

In [36]:
unique_countries = list(set(df["country"].apply(lambda x: str(x))))

unique_countries.sort()

for x in unique_countries:
    print(x)

Australia
Austria
Belgium
Canada
Channel Islands
Croatia
Czechoslovakia
Denmark
Faroe Islands
Finland
France
Germany
Gibraltar
Hungary
Iceland
India
Ireland
Isle of Man
Italy
Japan
Luxembourg
Mexico
Netherlands
New Zealand
Norway
Philippines
Poland
Romania
Russia
Samoa
South Africa
Spain
Sweden
Switzerland
Syria
Tonga
Turkiye
Ukraine
United Kingdom
United States
Zimbabwe
nan


In [21]:
# check nulls 
for i in range(len(df)):
    cur_country = str(df["country"].iloc[i])
    if cur_country == "" or cur_country == "nan":
        print(i + 2)
        print(df["birth_place"].iloc[i])
        print()

In [33]:
provinces = ["Alberta",
    "British Columbia",
    "Manitoba",
    "New Brunswick",
    "Newfoundland and Labrador",
    "Nova Scotia",
    "Ontario",
    "Prince Edward Island",
    "Quebec",
    "Saskatchewan",
    "Northwest Territories",
    "Nunavut",
    "Yukon"
]

ontario = ["Canada West", "Upper Canada"]

for i in range(len(df)):
    cur_country = str(df["country"].iloc[i])
    if cur_country == "Canada":
        cur_state = str(df["state"].iloc[i])
        if cur_state == "" or cur_state == "nan":
            cur_birthplace = str(df["birth_place"].iloc[i]).lower()
            for p in provinces:                
                if p.lower() in cur_birthplace:
                    df.loc[i, "state"] = p
                    break
            for p in ontario:
                if p.lower() in cur_birthplace:
                    df.loc[i, "state"] = "Ontario"
                    break
            


In [38]:
# check

for i in range(len(df)):
    cur_country = str(df["country"].iloc[i])
    if cur_country == "United Kingdom":
        cur_state = str(df["state"].iloc[i])
        if cur_state == "" or cur_state == "nan":
            # print(i + 2)
            print(i+2, df["birth_place"].iloc[i])
            # print()

4131 United Kingdom


In [39]:
import random

vals = random.sample(range(len(df)), k = 10)
for i in vals:
    print(i + 2)
    print(df["birth_place"].iloc[i])
    print(df["state"].iloc[i])
    print(df["country"].iloc[i])
    
    print()


7423
Hejringe, Birket, Lollands Nørre, Maribo, Denmark
nan
Denmark

992
Monroe, New York, United States
New York
United States

3908
Lind, Rind, Hammerum, Ringkøbing, Denmark
nan
Denmark

8078
Sutton, Yorkshire, England, United Kingdom
England
United Kingdom

8752
Caswell Township, Pender, North Carolina, United States
North Carolina
United States

6885
Middlewich, Cheshire, England
England
United Kingdom

9140
Alexander, Genesee, New York, United States
New York
United States

4439
Oneida, Center Township, Butler, Pennsylvania, United States
Pennsylvania
United States

9947
Gudme, Gudme, Gudme, Fyn, Denmark
nan
Denmark

8341
Rögla, Skårby, Ljunits härad, Malmöhus, Sweden
nan
Sweden



In [28]:
# write output

df.to_csv("birthplace_ai_cleaned.csv", index = False)

### Merge back to larger file

In [40]:
df = pd.read_csv("extra_info.csv")

df.head()

,name,birth_place,birth_place_state,birth_place_country,mission,loc_served,year,type,residence,url,familysearch_links,parent1_bio,parent2_bio,parent1_birthplace,parent2_birthplace
0,Lyman Palmer Pinkston,"Salt Lake City, Salt Lake, Utah",Utah,United States,Central States Mission,Central States,1931,Proselytizing,"Glendale, California",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZ7-BWD,When Everett Millard Pinkston was born on 4 Ju...,When Clara Margaret Palmer was born on 8 March...,"Bonne Terre, St. Francois, Missouri, United St...","Old Beaver, Clark, Idaho, United States"
1,Nelson Daniel Russ,"Alabama, Genesee, New York",New York,United States,Eastern States Mission,Eastern States,1898,Proselytizing,"Wilford, Fremont, Idaho, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/K2H8-5M4,"John Nelson Russ was born on 10 December 1818,...",When Abby Permelia Kenyon was born on 2 Decemb...,"Genesee, New York, United States","Charlestown, Washington, Rhode Island, United ..."
2,Preston Baxter Maughan,"Wellsville, Cache, Utah",Utah,United States,Northwestern States (Pacific) Mission,Northwestern States,1919,NaN,"Wellsville, Cache, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZX-BBP,When William Harrison Maughan Jr was born on 1...,When Margaret Love Baxter was born on 17 Febru...,"Wellsville, Cache, Utah, United States","Wellsville, Cache, Utah, United States"
3,Joseph Ezra Wood,"Holden, Millard, Utah",Utah,United States,British Mission,British,1912,NaN,"Holden, Millard, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWCK-W8D,When Charles Wood Jr. was born on 21 February ...,"When Sophia Dame was born on 31 March 1854, in...","Headcorn, Kent, England, United Kingdom","Fillmore, Millard, Utah, United States"
4,Jens Wilhelm Olsen,"Asnas, Oshered, Denmark",NaN,Denmark,Danish Mission,Danish,1935,NaN,United States,https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWJD-S8M,"When Jens Olsen was born on 23 September 1856,...",When Nicoline Pedersen was born on 13 March 18...,"Åstofte, Asnæs, Ods, Holbæk, Denmark","Asnæs, Ods, Holbæk, Denmark"


In [44]:
# parent 1 
parent1_df = pd.read_csv("parent2_no_duplicates.csv")

parent1_df.head()



,birth_place,state,country
0,"Old Beaver, Clark, Idaho, United States",Idaho,United States
1,"Charlestown, Washington, Rhode Island, United ...",Rhode Island,United States
2,"Wellsville, Cache, Utah, United States",Utah,United States
3,"Fillmore, Millard, Utah, United States",Utah,United States
4,"Asnæs, Ods, Holbæk, Denmark",NaN,Denmark


In [45]:
# Map birthplaces to 
parent1_birthplace_state = []
parent1_birthplace_country = []

for i in range(len(df)):
    cur_birthplace = df["parent2_birthplace"].iloc[i]
    mask = parent1_df['birth_place'] == cur_birthplace
    found_rows = parent1_df.loc[mask]
    if len(found_rows) > 0:
        parent1_birthplace_state.append(found_rows["state"].iloc[0])
        parent1_birthplace_country.append(found_rows["country"].iloc[0])
    else:
        parent1_birthplace_state.append("")
        parent1_birthplace_country.append("")

df["parent2_birthplace_state"] = parent1_birthplace_state
df["parent2_birthplace_country"] = parent1_birthplace_country


    




In [46]:
df.head()

,name,birth_place,birth_place_state,birth_place_country,mission,loc_served,year,type,residence,url,familysearch_links,parent1_bio,parent2_bio,parent1_birthplace,parent2_birthplace,parent1_birthplace_state,parent1_birthplace_country,parent2_birthplace_state,parent2_birthplace_country
0,Lyman Palmer Pinkston,"Salt Lake City, Salt Lake, Utah",Utah,United States,Central States Mission,Central States,1931,Proselytizing,"Glendale, California",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZ7-BWD,When Everett Millard Pinkston was born on 4 Ju...,When Clara Margaret Palmer was born on 8 March...,"Bonne Terre, St. Francois, Missouri, United St...","Old Beaver, Clark, Idaho, United States",Missouri,United States,Idaho,United States
1,Nelson Daniel Russ,"Alabama, Genesee, New York",New York,United States,Eastern States Mission,Eastern States,1898,Proselytizing,"Wilford, Fremont, Idaho, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/K2H8-5M4,"John Nelson Russ was born on 10 December 1818,...",When Abby Permelia Kenyon was born on 2 Decemb...,"Genesee, New York, United States","Charlestown, Washington, Rhode Island, United ...",New York,United States,Rhode Island,United States
2,Preston Baxter Maughan,"Wellsville, Cache, Utah",Utah,United States,Northwestern States (Pacific) Mission,Northwestern States,1919,NaN,"Wellsville, Cache, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZX-BBP,When William Harrison Maughan Jr was born on 1...,When Margaret Love Baxter was born on 17 Febru...,"Wellsville, Cache, Utah, United States","Wellsville, Cache, Utah, United States",Utah,United States,Utah,United States
3,Joseph Ezra Wood,"Holden, Millard, Utah",Utah,United States,British Mission,British,1912,NaN,"Holden, Millard, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWCK-W8D,When Charles Wood Jr. was born on 21 February ...,"When Sophia Dame was born on 31 March 1854, in...","Headcorn, Kent, England, United Kingdom","Fillmore, Millard, Utah, United States",England,United Kingdom,Utah,United States
4,Jens Wilhelm Olsen,"Asnas, Oshered, Denmark",NaN,Denmark,Danish Mission,Danish,1935,NaN,United States,https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWJD-S8M,"When Jens Olsen was born on 23 September 1856,...",When Nicoline Pedersen was born on 13 March 18...,"Åstofte, Asnæs, Ods, Holbæk, Denmark","Asnæs, Ods, Holbæk, Denmark",NaN,Denmark,NaN,Denmark


In [47]:
df.to_csv("all_clean.csv", index = False)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48349 entries, 0 to 48348
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   name                        48349 non-null  object
 1   birth_place                 48197 non-null  object
 2   birth_place_state           44025 non-null  object
 3   birth_place_country         48197 non-null  object
 4   mission                     48349 non-null  object
 5   loc_served                  43361 non-null  object
 6   year                        48349 non-null  int64 
 7   type                        30108 non-null  object
 8   residence                   47051 non-null  object
 9   url                         48349 non-null  object
 10  familysearch_links          48159 non-null  object
 11  parent1_bio                 48026 non-null  object
 12  parent2_bio                 48039 non-null  object
 13  parent1_birthplace          47994 non-null  ob

### Other things

In [42]:
new_df = df.drop_duplicates()

In [43]:
len(new_df)

11347

In [44]:
len(set(new_df["birth_place"]))

11347

In [61]:
new_df.head(20)

,birth_place,state,country
0,"Salt Lake City, Salt Lake, Utah",Utah,United States
1,"Alabama, Genesee, New York",New York,United States
2,"Wellsville, Cache, Utah",Utah,United States
3,"Holden, Millard, Utah",Utah,United States
4,"Asnas, Oshered, Denmark",NaN,Denmark
5,"Fillmore, Millard, Utah",Utah,United States
6,"Herriman, Salt Lake, Utah",Utah,United States
7,"Santaquin, Utah, Utah",Utah,United States
8,"Grantsville, Tooele, Utah",Utah,United States
9,"Washington, Washington, Utah",Utah,United States


In [65]:
new_df["country"].iloc[1]

'United States'

In [45]:
data_df = pd.read_csv("initial_clean.csv")

In [55]:
cleaned_vals = set(new_df["birth_place"])
count = 0

items = []

for i in range(len(data_df)):
    cur_birthplace = data_df["birth_place"].iloc[i]
    if not cur_birthplace in cleaned_vals and not str(cur_birthplace) == "nan":
        val = cur_birthplace.split()[-1].strip()
        if "," in val:
            val = val.replace(",", "")
        if val == "Danmark" or val == "DÃ\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã\x82Â¤nemark":
            val = "Denmark"
        if val == "Deutschland" or val == "Prussia":
            val = "Germany"
        if val == "States":
            print(cur_birthplace)
            
            val = "United States"
        if val == "Ã\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã¢Â\x80Â\x93sterreich":
            val = "Austria"
        if val == "MÃ\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã\x82Â©xico":
            val = "Mexico"
        if val == "Kingdom":            
            print(cur_birthplace)
            val = "United Kingdom"
        items.append(val)
print(set(items))

Fellos, DoÃÂÃÂÃÂÃÂ±a Ana, New Mexico, United States
CwmbrÃÂÃÂÃÂÃÂ¢n, Monmouthshire, Wales, United Kingdom
RÃÂÃÂÃÂÃÂ¡th MaolÃÂÃÂÃÂÃÂ¡in, County Donegal, Ireland, United Kingdom
{'Norway', 'Belgium', 'Denmark', 'Spain', 'Switzerland', 'Sweden', 'United States', 'Iceland', 'Finland', 'United Kingdom', 'Austria', 'India', 'Mexico', 'Germany'}


In [66]:
# create mapping

birthplace_mapping = {}
for i in range(len(new_df)):
    place = new_df["birth_place"].iloc[i]
    state = new_df["state"].iloc[i]
    country = new_df["country"].iloc[i]
    
    birthplace_mapping[place] = (state, country)

cleaned_vals = set(new_df["birth_place"])

for i in range(len(data_df)):
    cur_birthplace = data_df["birth_place"].iloc[i]
    state = ""
    country = ""
    if not cur_birthplace in birthplace_mapping and not str(cur_birthplace) == "nan":
        val = cur_birthplace.split()[-1].strip()
        if "," in val:
            val = val.replace(",", "")
        if val == "Danmark" or val == "DÃ\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã\x82Â¤nemark":
            val = "Denmark"
        if val == "Deutschland" or val == "Prussia":
            val = "Germany"
        if val == "States":
            new_vals = cur_birthplace.split(", ")
            state = new_vals[-2]
            val = "United States"
        if val == "Ã\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã¢Â\x80Â\x93sterreich":
            val = "Austria"
        if val == "MÃ\x83Â\x83Ã\x86Â\x92Ã\x83Â\x82Ã\x82Â©xico":
            val = "Mexico"
        if val == "Kingdom":            
            new_vals = cur_birthplace.split(", ")
            state = new_vals[-2]
            val = "United Kingdom"
        country = val
    elif cur_birthplace in birthplace_mapping:
        state, country = birthplace_mapping[cur_birthplace]
    birthplace_mapping[cur_birthplace] = (state, country)

In [60]:
birthplace_mapping

{'Salt Lake City, Salt Lake, Utah': ('', ''),
 'Alabama, Genesee, New York': ('', ''),
 'Wellsville, Cache, Utah': ('', ''),
 'Holden, Millard, Utah': ('', ''),
 'Asnas, Oshered, Denmark': ('', ''),
 'Fillmore, Millard, Utah': ('', ''),
 'Herriman, Salt Lake, Utah': ('', ''),
 'Santaquin, Utah, Utah': ('', ''),
 'Grantsville, Tooele, Utah': ('', ''),
 'Washington, Washington, Utah': ('', ''),
 'Morgan, Morgan, Utah, United States': ('', ''),
 'Echo, Summit, Utah': ('', ''),
 'Salt Lake City, Utah': ('', ''),
 'Salt Lake City, Salt Lake, Utah, United States': ('', ''),
 'Salt Lake City, Salt Lake, Utah Territory, United States': ('', ''),
 'Manti, Sanpete, Utah': ('', ''),
 'Ogden, Weber, Utah': ('', ''),
 'Corinne Box Elder, Utah, United States': ('', ''),
 'Provo, Utah, Utah Territory, United States': ('', ''),
 'Saint George, Washington, Utah': ('', ''),
 'Jerslev, HjÃƒÂƒÃ‚ÂƒÃƒÂ†Ã‚Â’ÃƒÂƒÃ‚Â‚ÃƒÂ‚Ã‚Â¸rring, Denmark': (nan, 'Denmark'),
 'Ithaca, Thompkins, New York, United States': ('',

In [67]:
# the moment of truth
new_states = []
new_countries = []

for i in range(len(data_df)):
    cur_birthplace = data_df["birth_place"].iloc[i]
    if cur_birthplace in birthplace_mapping:
        state, country = birthplace_mapping[cur_birthplace]
    else:
        state = ""
        country = ""
    new_states.append(state)
    new_countries.append(country)
    
data_df["birth_place_state"] = new_states
data_df["birth_place_country"] = new_countries

data_df.head(20)

,name,birth_place,mission,loc_served,year,type,residence,url,familysearch_links,parent1_bio,parent2_bio,parent1_name,parent1_birthplace,parent2_name,parent2_birthplace,birth_place_state,birth_place_country
0,Lyman Palmer Pinkston,"Salt Lake City, Salt Lake, Utah",Central States Mission,Central States,1931,Proselytizing,"Glendale, California",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZ7-BWD,When Everett Millard Pinkston was born on 4 Ju...,When Clara Margaret Palmer was born on 8 March...,Everett Millard Pinkston,"Bonne Terre, St. Francois, Missouri, United St...",Clara Margaret Palmer,"Old Beaver, Clark, Idaho, United States",Utah,United States
1,Nelson Daniel Russ,"Alabama, Genesee, New York",Eastern States Mission,Eastern States,1898,Proselytizing,"Wilford, Fremont, Idaho, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/K2H8-5M4,"John Nelson Russ was born on 10 December 1818,...",When Abby Permelia Kenyon was born on 2 Decemb...,Nelson Russ,"Genesee, New York, United States",Abby Permelia Kenyon,"Charlestown, Washington, Rhode Island, United ...",New York,United States
2,Preston Baxter Maughan,"Wellsville, Cache, Utah",Northwestern States (Pacific) Mission,Northwestern States,1919,NaN,"Wellsville, Cache, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWZX-BBP,When William Harrison Maughan Jr was born on 1...,When Margaret Love Baxter was born on 17 Febru...,William Harrison Maughan Jr,"Wellsville, Cache, Utah, United States",Margaret Love Baxter,"Wellsville, Cache, Utah, United States",Utah,United States
3,Joseph Ezra Wood,"Holden, Millard, Utah",British Mission,British,1912,NaN,"Holden, Millard, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWCK-W8D,When Charles Wood Jr. was born on 21 February ...,"When Sophia Dame was born on 31 March 1854, in...",Charles Wood Jr.,"Headcorn, Kent, England, United Kingdom",Sophia Dame,"Fillmore, Millard, Utah, United States",Utah,United States
4,Jens Wilhelm Olsen,"Asnas, Oshered, Denmark",Danish Mission,Danish,1935,NaN,United States,https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWJD-S8M,"When Jens Olsen was born on 23 September 1856,...",When Nicoline Pedersen was born on 13 March 18...,Jens Olsen,"Åstofte, Asnæs, Ods, Holbæk, Denmark",Nicoline Pedersen,"Asnæs, Ods, Holbæk, Denmark",NaN,Denmark
5,Melvin Porter Callister,"Fillmore, Millard, Utah",Northwestern States (Pacific) Mission,Northwestern States,1919,NaN,"Salt Lake City, Salt Lake, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KW8J-C74,When Daniel Porter Callister was born on 28 De...,When Melissa Elvira Davis was born on 31 Octob...,Daniel Porter Callister,"Salt Lake City, Salt Lake, Utah, United States",Melissa Elvira Davis,"Salt Lake City, Salt Lake, Utah, United States",Utah,United States
6,Emma Virginia Bodell,"Herriman, Salt Lake, Utah",Northwestern States (Pacific) Mission,Northwestern States,1913,Proselytizing,"Herriman, Salt Lake, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/KWCK-X76,When Joseph Samuel Henry Bodell was born on 23...,When Sarah Lovinia Howard was born on 16 Janua...,Joseph Samuel Henry Bodell,"Herriman, Salt Lake, Utah, United States",Sarah Lovinia Howard,"Millcreek, Salt Lake, Utah, United States",Utah,United States
7,Jedediah P Nelson,"Santaquin, Utah, Utah",Northern States Mission,Northern States,1900,NaN,"Santaquin, Utah, Utah, United States",https://history.churchofjesuschrist.org/chd/in...,https://ancestors.familysearch.org/en/K24B-PPR,"When Neils Nelson was born on 20 August 1839, ...",When Eleanor Jepson Openshaw was born on 15 Ap...,Neils Nelson,"Villie, Ljunits härad, Malmöhus, Sweden",Eleanor Jepson Opensha

In [68]:
data_df.to_csv("clean_birthplace.csv", index=False)